In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pickle
import pandas as pd

import networkx as nx

import matplotlib.pyplot as plt

from matplotlib.colors import PowerNorm
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

from matrix_processing_helpers import sparsify_global_percentile
from plot_helpers import plot_rollout_graph, plot_feature_ranking_comparison

In [2]:
with open("data_processed/dataframes/input_cols_pruned.pkl", "rb") as f:
    INPUT_COLS_PRUNED = pickle.load(f)

feature_names = list(INPUT_COLS_PRUNED)

### Attention

For each given number of layers $L$ we have attention matrices $\{A^{(\ell)}\}_{\ell=1}^L$.
We use them to compute the rollout matrix 
$$ R = \hat A^{(1)} \hat A^{(2)} \cdot \hat A^{(L)}\in\mathbb{R}^{d\times d}$$

where $\hat A^{(\ell)}$ is the row-normalization of $A^{(\ell)}$. We then compute the **attention-based feature importance scores** summing over the columns of $R$:
$$ s_j = \sum_{i=1}^d R_{ij}, \quad j\in\{1,\ldots,d\}$$


In [3]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 

for layer in NUM_LAYERS:
    rollouts = np.load(f"Results/rollout_global/{layer}layers/rollout{layer}layers.npz")

    R = rollouts['rollout']
    
    importance = R.sum(axis=0)    
    df = pd.DataFrame({'feature': feature_names, 
                       'score': importance,      
                       'original_position': range(len(feature_names))
                      })

    df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)    
    df_sorted['ranking'] = np.arange(1, len(feature_names)+1)

    df.to_csv(f"Results/attention_scores/{layer}layers/attention_scores{layer}layers_unsorted.csv", 
                     index=False, float_format="%.6f")
    
    df_sorted.to_csv(f"Results/attention_scores/{layer}layers/attention_scores{layer}layers.csv", 
                     index=False, float_format="%.6f")

### Attention + values

We correct the attention-based feature importance scores by accounting for the effect of the value matrices.

For each given number of layers $L$ we have value matrices $\{V^{(\ell)}\}_{\ell=1}^L$.
We use them to compute correction weights
$$ v_j = \frac{1}{L}\sum_{\ell=1}^L \|V^{(\ell)}_{j\bullet}\|_2$$

where $\hat V^{(\ell)}_{j\bullet}\in\mathbb{R}^d$ is the $j$-th row of $V^{(\ell)}$. We then compute the **value-based feature importance scores** as:
$$ \mathcal{S}_j = s_j v_j, \quad j\in\{1,\ldots,d\}$$

In [4]:
for layer in NUM_LAYERS:
    rollouts = np.load(f"Results/rollout_global/{layer}layers/rollout{layer}layers.npz")
    R = rollouts['rollout']
    importance = R.sum(axis=0)

    with open(f"Results/rollout_local/{layer}layers/value_matrices{layer}layers.pkl", 
              "rb") as f:
        val_matrices = pickle.load(f)
    
    val_list = [np.mean(val_matrices[key].numpy(), axis=0) for key in val_matrices.keys()]
    val_scores = [np.linalg.norm(matrix, axis=1) for matrix in val_list]
    val_scores_all = np.mean(val_scores, axis=0)

    scores = val_scores_all*importance
    
    df = pd.DataFrame({'feature': feature_names, 
                       'score': scores, 
                       'original_position': range(len(feature_names))
                      })

    df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)
    df_sorted['ranking'] = np.arange(1, len(feature_names)+1)

    df.to_csv(f"Results/attention_scores/{layer}layers/value_scores{layer}layers_unsorted.csv", 
                     index=False, float_format="%.6f")

    df_sorted.to_csv(f"Results/attention_scores/{layer}layers/value_scores{layer}layers.csv", 
                     index=False, float_format="%.6f")

### SPECTRAL

Importance scores based on PCA computed as
$$I_i = \sum_{k=1}^d \sigma_k^s|v_{ki}|^2, \quad i=1,\ldots,d$$

In [5]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 

for layer in NUM_LAYERS:
    rollouts = np.load(f"Results/rollout_global/{layer}layers/rollout{layer}layers.npz")
    R = rollouts['rollout']

    U, S, Vh = np.linalg.svd(R, full_matrices=False)

    scores = np.sum((S[:, None] ** 2) * (np.abs(Vh) ** 2), axis=0)

    df_pca = pd.DataFrame({'feature': feature_names,
                           'score': scores,
                           'original_position': range(len(feature_names))
                           })

    df_sorted_pca = df_pca.sort_values(by='score', ascending=False).reset_index(drop=True)
    df_sorted_pca['ranking'] = np.arange(1, len(feature_names)+1)

    df_pca.to_csv(f"Results/attention_scores/{layer}layers/pca_scores{layer}layers_unsorted.csv", 
                         index=False, float_format="%.6f")

    df_sorted_pca.to_csv(f"Results/attention_scores/{layer}layers/pca_scores{layer}layers.csv", 
                         index=False, float_format="%.6f")

### SHAP

In [6]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 

for layer in NUM_LAYERS:
    shap_vals = np.load(f"Results/shap_values/values/{layer}layers/shap_values_{layer}layers.npz")
    shap_importance = np.mean(np.abs(shap_vals["shap_vals"]), axis=0)

    df_shap = pd.DataFrame({'feature': feature_names,
                            'score': shap_importance,
                            'original_position': range(len(feature_names))
                           })

    df_sorted_shap = df_shap.sort_values(by='score', ascending=False).reset_index(drop=True)
    df_sorted_shap['ranking'] = np.arange(1, len(feature_names)+1)

    df_shap.to_csv(f"Results/shap_values/tables/{layer}layers/shap_scores{layer}layers_unsorted.csv", 
          index=False, float_format="%.6f")

    df_sorted_shap.to_csv(f"Results/shap_values/tables/{layer}layers/shap_scores{layer}layers.csv", 
          index=False, float_format="%.6f")